# Lot 3 : Assemblage "Memory-Efficient" (approche Minia)

Ce notebook montre l'assemblage sans construire le graphe de de Bruijn en mémoire :
1. on garde les k-mers solides,
2. on les insère dans un **filtre de Bloom**,
3. on parcourt le graphe **implicite** en testant les 4 extensions A/C/G/T,
4. on reconstruit les contigs et on compare à la référence (identité en %).

Le critère de recette : reconstruire la séquence cible avec une identité > 98%.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.io_sequences import lire_fastq, lire_fasta
from src.assemblage import assembler, meilleure_identite, identite

if not os.path.exists("../data/toy_reads.fastq"):
    from src.generer_toy import generer_toy
    generer_toy()

reads = lire_fastq("../data/toy_reads.fastq")
reference = lire_fasta("../data/reference.fasta")[0][1]

print("Nombre de reads :", len(reads))
print("Longueur de la reference :", len(reference), "bases")

Nombre de reads : 2000
Longueur de la reference : 400 bases


## 1. Assemblage avec les paramètres conseillés

Pour le toy dataset : `k = 21` et `seuil = 5` (pour filtrer les k-mers issus d'erreurs).

In [ ]:
k = 21
seuil = 5

contigs = assembler(reads, k, seuil=seuil)

print("Nombre de contigs produits :", len(contigs))
print("Longueur du plus long contig :", len(contigs[0]))
print()
ident = meilleure_identite(contigs, reference)
print("Meilleure identite avec la reference : %.2f%%" % ident)
print("Critere de recette (> 98%) atteint :", ident > 98)

Nombre de contigs produits : 1
Longueur du plus long contig : 399

Meilleure identite avec la reference : 99.75%
Critere de recette (> 98%) atteint : True


In [ ]:
# Apercu du plus long contig
contig = contigs[0]
print("Debut du contig :", contig[:60], "...")
print()
print("Debut de la reference :", reference[:60], "...")

Debut du contig : AGCCCAATAAACCACTCTGACTGGCCGAATAGGGATATAGGCAACGACATGTGCGGCGAC ...

Debut de la reference : AAGCCCAATAAACCACTCTGACTGGCCGAATAGGGATATAGGCAACGACATGTGCGGCGA ...


## 2. Influence de la taille de k et du seuil

On fait varier `k` et le `seuil` pour voir leur impact sur la qualité de l'assemblage.
- Un `k` trop petit crée des ambiguïtés (répétitions), un `k` trop grand est sensible aux erreurs.
- Un `seuil` trop bas laisse passer des erreurs, un `seuil` trop haut supprime de vrais k-mers.

In [ ]:
print("k  | seuil | nb contigs | max len | identite")
print("-" * 50)
for k_test in [15, 21, 27]:
    for seuil_test in [2, 5, 8]:
        contigs_test = assembler(reads, k_test, seuil=seuil_test)
        if len(contigs_test) > 0:
            ident_test = meilleure_identite(contigs_test, reference)
            max_len = len(contigs_test[0])
        else:
            ident_test = 0.0
            max_len = 0
        print("%2d | %5d | %10d | %7d | %.1f%%" % (k_test, seuil_test, len(contigs_test), max_len, ident_test))

k  | seuil | nb contigs | max len | identite
--------------------------------------------------


15 |     2 |         41 |      80 | 20.0%
15 |     5 |          1 |     399 | 99.8%
15 |     8 |          1 |     397 | 99.2%


21 |     2 |         31 |     135 | 33.8%
21 |     5 |          1 |     399 | 99.8%


21 |     8 |          1 |     397 | 99.2%


27 |     2 |         18 |     214 | 53.5%
27 |     5 |          1 |     399 | 99.8%


27 |     8 |          1 |     397 | 99.2%
